# Pose Estimator Training

This notebook is used to fine-tune the YOLO11 object-detection model for pose estimation in the context of long-distance running. YOLO11 is a verstile computer vision model with variants for detection, instance segmentation, and pose estimation released in September 2024. 

In [31]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("MPS device found.")
else:
    print("MPS device not found.")

MPS device found.


In [32]:
from ultralytics import YOLO

# model size options: 'n', 's', 'm', 'l', 'x' in increasing size
model_n = YOLO('yolo11n-pose.pt').to(device)

## Training YOLO11

This function is a wrapper for the existing Ultralytics function to train a model. 

In [33]:
project_ = 'running_project'

def train_model(model, epochs=5, project=project_, name='default'):
    if name == 'default':
        print("Error: provide a model name to avoid overwriting")
        return
    
    print("Fine-tuning start...")
    results = model.train(
        data='datasets/athlete_pose_converted/athlete_pose.yaml',
        epochs=epochs,
        imgsz=640, # 640 is YOLO standard resolution
        batch=16,
        device=torch.device("mps"),
        project=project,
        name=name,
        plots=True,
        save=True
    )

    print(f"Model weights: {project}/{name}/weights/best.pt")

    return results

## Evaluating the Model

Ultralytics also provides a function for evaluation. This evaluates model performance on the validation set.

The metrics used to evaluate the model are mAP50-95 for the bounding box and pose. 

In [34]:
def evaluate_model(model=None, project=project_, name='default'):
    if model is None:
        if name != 'default':
            custom_model_path = f"runs/pose/{project}/{name}/weights/best.pt"
            print(f"Loading custom model from {custom_model_path}...")
            model = YOLO(custom_model_path)
        else:
            print("Error: provide a model name to load model")
            return
        
    if model is not None and name == 'default':
        print("Error: provide a model name to load model")
        return


    print("Running validation pipeline...")
    metrics = model.val(
        data='datasets/athlete_pose_converted/athlete_pose.yaml',
        split='val',
        project=project,
        name=name
    )

    print("\n--- Evaluation Results ---")
    
    # mAP50-95 is the standard accuracy metric for pose estimation 
    map_score = metrics.box.map    # Bounding box mAP
    pose_map = metrics.pose.map    # Pose keypoint mAP
    
    print(f"Bounding Box mAP50-95: {map_score:.4f}")
    print(f"Pose Keypoint mAP50-95: {pose_map:.4f}")
    
    print(f"\nDetailed metric plots (confusion matrices, PR curves) have been saved to: {project}/{name}/")

In [10]:
base_model_n_results = evaluate_model(model_n)

Running validation pipeline...
val: Fast image access ✅ (ping: 0.2±0.1 ms, read: 507.6±189.9 MB/s, size: 170.2 KB)
val: Scanning /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose_converted/valid/labels... 3108 images, 0 backgrounds, 73 corrupt: 100% ━━━━━━━━━━━━ 3108/3108 2.9Kit/s 1.1s0.1s
val: /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose_converted/valid/images/20000000018.jpg: ignoring corrupt image/label: negative class labels or coordinate [-0.0126094]
val: /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose_converted/valid/images/20000000019.jpg: ignoring corrupt image/label: negative class labels or coordinate [-0.0124531 -0.0187109 -0.0179609 -0.0170078]
val: /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose_converted/valid/images/20000000020.jpg: ignoring corrupt image/label: negative class labels or coordinate [-0.0255625 -0.0273359 -0.0331641 -0.0159297 -0.0290703 -0.0331719]

In [ ]:
tuned_model_n = train_model(model_n, epochs=5, name='1_model_n_epoch5')

Fine-tuning start...
New https://pypi.org/project/ultralytics/8.4.38 available 😃 Update with 'pip install -U ultralytics'
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets/athlete_pose_converted/athlete_pose.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=tuned_model_n_5epoch, nbs=64, nms=False,

In [3]:
project_ = "running_project"

def fine_tune_model(model, freeze=10, epochs=5, project=project_, name='default'):
    if name == 'default':
        print("Error: provide a model name to avoid overwriting")
        return
    
    print("Fine-tuning start...")
    results = model.train(
        data='datasets/athlete_pose_converted/athlete_pose.yaml',
        epochs=epochs,
        imgsz=640, # 640 is YOLO standard resolution
        batch=16,
        dropout=0.2,
        device=torch.device("mps"),
        freeze=freeze,
        project=project,
        name=name,
        plots=True,
        save=True
    )

    print(f"Model weights: {project}/{name}/weights/best.pt")

    return results

In [36]:
model_n2 = YOLO('yolo11n-pose.pt').to(device)
fine_tuned_model_n = fine_tune_model(model_n2, epochs=5, name='2_model_n_epoch5_freeze10')

Fine-tuning start...
New https://pypi.org/project/ultralytics/8.4.41 available 😃 Update with 'pip install -U ultralytics'
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets/athlete_pose_converted/athlete_pose.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.2, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=2_model_n_epoch5_freeze102, nbs=64, nms=Fa

In [37]:
retrained_model_n_results = evaluate_model(name='2_model_n_epoch5_freeze102')

Loading custom model from runs/pose/running_project/2_model_n_epoch5_freeze102/weights/best.pt...
Running validation pipeline...
Ultralytics 8.4.37 🚀 Python-3.14.2 torch-2.11.0 CPU (Apple M1 Pro)
YOLO11n-pose summary (fused): 110 layers, 2,866,468 parameters, 0 gradients, 7.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 411.3±71.8 MB/s, size: 162.7 KB)
val: Scanning /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose_converted/valid/labels.cache... 3108 images, 0 backgrounds, 73 corrupt: 100% ━━━━━━━━━━━━ 3108/3108 103.5Mit/s 0.0s
val: /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose_converted/valid/images/20000000018.jpg: ignoring corrupt image/label: negative class labels or coordinate [-0.0126094]
val: /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose_converted/valid/images/20000000019.jpg: ignoring corrupt image/label: negative class labels or coordinate [-0.0124531 -0.0187109 -0.0179609 -0.017007

In [26]:
fine_tuned_model_n_results = evaluate_model(name='fine_tuned_model_n_5epoch')

Loading custom model from runs/pose/running_project/fine_tuned_model_n_5epoch/weights/best.pt...
Running validation pipeline...
Ultralytics 8.4.37 🚀 Python-3.14.2 torch-2.11.0 CPU (Apple M1 Pro)
YOLO11n-pose summary (fused): 110 layers, 2,866,468 parameters, 0 gradients, 7.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 468.9±93.9 MB/s, size: 172.6 KB)
val: Scanning /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose_converted/valid/labels.cache... 3108 images, 0 backgrounds, 73 corrupt: 100% ━━━━━━━━━━━━ 3108/3108 501.4Mit/s 0.0s
val: /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose_converted/valid/images/20000000018.jpg: ignoring corrupt image/label: negative class labels or coordinate [-0.0126094]
val: /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose_converted/valid/images/20000000019.jpg: ignoring corrupt image/label: negative class labels or coordinate [-0.0124531 -0.0187109 -0.0179609 -0.0170078

# Fine-Tuned Model Performance

YOLO11 Nano performance: base vs fine-tuned (5 epochs)

2,866,468 parameters

batch size: 16

| Metric | Base     | Fine-tuned (freeze backbone) | Fine-tuned (unfrozen) |
| ---- | -------- | -------- | ------ |
| Bounding Box Accuracy   | 0.8615  | 0.9836 | 0.9801 |
| Pose Keypoint Accuracy  | 0.5560   | 0.7534 | 0.7610 |
| Training Time (mins)       | - | 48  | 106  |
| Training Memory (GB)    | - | 4.35| 5.37 |



Model S:

YOLO11s-pose summary (fused): 109 layers, 9,902,940 parameters

In [ ]:
model_s = YOLO('yolo11s-pose.pt').to(device)

In [28]:
model_s_results = evaluate_model(model_s, name='3_model_s_base')

Running validation pipeline...
YOLO11s-pose summary (fused): 109 layers, 9,902,940 parameters, 0 gradients, 23.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 363.5±95.3 MB/s, size: 157.3 KB)
val: Scanning /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose_converted/valid/labels.cache... 3108 images, 0 backgrounds, 73 corrupt: 100% ━━━━━━━━━━━━ 3108/3108 155.2Mit/s 0.0s
val: /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose_converted/valid/images/20000000018.jpg: ignoring corrupt image/label: negative class labels or coordinate [-0.0126094]
val: /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose_converted/valid/images/20000000019.jpg: ignoring corrupt image/label: negative class labels or coordinate [-0.0124531 -0.0187109 -0.0179609 -0.0170078]
val: /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose_converted/valid/images/20000000020.jpg: ignoring corrupt image/label: negative cl



| Metric | Base Small     | Fine-tuned nano (freeze backbone) |
| ---- | -------- | -------- |
| Bounding Box Accuracy   | 0.8663  | 0.9836 |
| Pose Keypoint Accuracy  | 0.6168  | 0.7534 |
| Parameters | 9,902,940 | 2,866,468 |


Bounding Box mAP50-95: 0.8663
Pose Keypoint mAP50-95: 0.6168

Discuss tradeoffs between model sizes and choice for fine-tuning final model.

In [29]:
fine_tuned_model_s = fine_tune_model(model_s, epochs=10, freeze=10, name='4_model_s_epoch10_freeze10')

Fine-tuning start...
New https://pypi.org/project/ultralytics/8.4.38 available 😃 Update with 'pip install -U ultralytics'
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets/athlete_pose_converted/athlete_pose.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=4_model_s_epoch10_freeze10, nbs=64, nms=F

In [30]:
fine_tuned_model_s_results = evaluate_model(name='4_model_s_epoch10_freeze10')

Loading custom model from runs/pose/running_project/4_model_s_epoch10_freeze10/weights/best.pt...
Running validation pipeline...
Ultralytics 8.4.37 🚀 Python-3.14.2 torch-2.11.0 CPU (Apple M1 Pro)
YOLO11s-pose summary (fused): 110 layers, 9,902,940 parameters, 0 gradients, 23.1 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.0 ms, read: 611.6±246.9 MB/s, size: 155.5 KB)
val: Scanning /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose_converted/valid/labels.cache... 3108 images, 0 backgrounds, 73 corrupt: 100% ━━━━━━━━━━━━ 3108/3108 18.3Mit/s 0.0s
val: /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose_converted/valid/images/20000000018.jpg: ignoring corrupt image/label: negative class labels or coordinate [-0.0126094]
val: /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose_converted/valid/images/20000000019.jpg: ignoring corrupt image/label: negative class labels or coordinate [-0.0124531 -0.0187109 -0.0179609 -0.01700

| Metric | Fine-tuned nano (freeze backbone) | Fine-tuned small (freeze backbone) |
| ---- | -------- | -------- |
| Bounding Box Accuracy   | 0.9836 | 0.9874 |
| Pose Keypoint Accuracy  | 0.7534 | 0.7743 |
| Parameters | 2,866,468 | 9,902,940 |
| Training time (mins) | 48 | 401 |
| GPU Memory (GB) | 4.35 | 5.51 |